# 影線、量能與價位區域：重訓回測驗收
本機固定歷史的離線研究。只輸出彙總、來源hash與檢查，不含帳務、憑證或原始行情。以CPython順序實際執行，非Jupyter kernel。後段已重用；數值為成本假設下模擬。

In [1]:
from pathlib import Path
import hashlib,json,sys
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'server/fubon_bridge').is_dir())
sys.path.insert(0,str(root/'server'))
from fubon_bridge.research import digest
from fubon_bridge.intraday_strategy import futures_tax
folder=root/'.fubon-dev/reports/wick-volume-zones-v1-verified'
frozen=json.loads((folder/'frozen.json').read_text())
assert digest({k:v for k,v in frozen.items() if k!='id'})==frozen['id']
for file,sha in frozen['implementation'].items():
    assert hashlib.sha256((root/file).read_bytes()).hexdigest()==sha,file
    assert hashlib.sha256((folder/'source'/file).read_bytes()).hexdigest()==sha,file
aux=json.loads((folder/'auxiliary.json').read_text())
assert aux['record']==frozen['auxiliary'] and digest(aux['model'])==aux['record']['hash']
assert aux['record']['metadata']['maximumLabelEnd']<aux['record']['cutoff']
print(json.dumps(dict(frozenId=frozen['id'],sourceFiles=len(frozen['implementation']),auxiliaryCutoff=aux['record']['cutoff'],executionAllowed=False,adoptionAllowed=False),ensure_ascii=False))


{"frozenId": "9052412cd5eb71d5f3a6f7c7d2c5fb2013f61358155fcde946b9c1bafbbf2097", "sourceFiles": 71, "auxiliaryCutoff": 1727280000, "executionAllowed": false, "adoptionAllowed": false}


In [2]:
rows=[];checks=0
for session in ('regular','afterhours'):
    report=json.loads((folder/f'{session}-evaluation.json').read_text())
    assert report['frozenId']==frozen['id']
    assert report['selected'] is None and not report['numericQualified']
    assert not report['executionAllowed'] and not report['adoptionAllowed']
    assert report['quality']['parentControl']['excludedParentEvents']==0
    for policy,phases in report['results'].items():
        for phase,comparison in phases['raw'].items():
            for scenario,metrics in comparison.items():
                trades=metrics['trades'];assert len(trades)==metrics['count']
                assert abs(sum(t['net'] for t in trades)-metrics['net'])<1e-7
                assert abs(sum(metrics['dates'].values())-metrics['net'])<1e-7
                by_day={}
                for t in trades:
                    side=1 if t['side']=='Buy' else -1
                    tax=futures_tax(t['entry'],10,.00002)+futures_tax(t['exit'],10,.00002)
                    assert abs((t['exit']-t['entry'])*side*10-40-tax-t['net'])<1e-7
                    assert float(t['entry']).is_integer() and float(t['exit']).is_integer()
                    previous=by_day.setdefault(t['day'],[])
                    assert len(previous)<2 and all(p['side']!=t['side'] and p['exitTime']<=t['time'] for p in previous)
                    realized=sum(p['net'] for p in previous)
                    assert t['risk']<=min(1000,1000+realized)+1e-7
                    previous.append(t)
                checks+=1
            normal=comparison['normal'];nets=[t['net'] for t in normal['trades']]
            interval=phases['dailyIntervals'][phase]['normal']
            rows.append(dict(session=session,policy=policy,phase=phase,count=normal['count'],net=normal['net'],
                stressNet=comparison['stress']['net'],delayedNet=comparison['delayed']['net'],maxDrawdown=normal['maxDrawdown'],
                meanTrade=float(np.mean(nets)) if nets else None,medianTrade=float(np.median(nets)) if nets else None,
                dailyMean=interval['mean'],dailyMeanLower=interval['lower'],dailyMeanUpper=interval['upper']))
print(json.dumps(dict(comparisonsChecked=checks,allSelected=None,adoptionAllowed=False,rows=rows),ensure_ascii=False,indent=2))


{
  "comparisonsChecked": 72,
  "allSelected": null,
  "adoptionAllowed": false,
  "rows": [
    {
      "session": "regular",
      "policy": "confirmed-rule",
      "phase": "validation",
      "count": 53,
      "net": 1126.0,
      "stressNet": -62.0,
      "delayedNet": 5442.0,
      "maxDrawdown": 3848.0,
      "meanTrade": 21.245283018867923,
      "medianTrade": -82.0,
      "dailyMean": 10.622641509433961,
      "dailyMeanLower": -82.45566037735848,
      "dailyMeanUpper": 96.252358490566
    },
    {
      "session": "regular",
      "policy": "confirmed-rule",
      "phase": "historical_reused_test",
      "count": 25,
      "net": -13594.0,
      "stressNet": -12966.0,
      "delayedNet": -5129.0,
      "maxDrawdown": 13594.0,
      "meanTrade": -543.76,
      "medianTrade": -820.0,
      "dailyMean": -128.24528301886792,
      "dailyMeanLower": -220.64316037735847,
      "dailyMeanUpper": -42.28797169811323
    },
    {
      "session": "regular",
      "policy": "confirme

In [3]:
model_groups=heads=missing=0
for path in sorted(folder.glob('*-model.json')):
    model=json.loads(path.read_text())
    if model is None:missing+=1;continue
    assert model['kind']=='zone-entry-value' and len(model['features'])==90
    assert set(model['heads'])=={'normal','stress','delayed'}
    model_groups+=1;heads+=len(model['heads'])
for session,spec in frozen['sessions'].items():
    for name,record in spec['models'].items():
        model=json.loads((folder/record['file']).read_text())
        assert digest(model)==record['hash']
        meta=record['metadata']
        if model is not None:
            assert meta['fitActions']>=100 and meta['fitDays']>=20
            assert meta['maximumLabelEnd']<meta['calibrationCutoff']
print(json.dumps(dict(nativeModelGroups=model_groups,nativeHeads=heads,insufficientModelGroups=missing,
    reuseHistory=True,allChecksPassed=True),ensure_ascii=False))


{"nativeModelGroups": 13, "nativeHeads": 39, "insufficientModelGroups": 11, "reuseHistory": true, "allChecksPassed": true}
